# 3.7 · 文本特征工程 / Text Feature Engineering

> **课程定位 / Where this fits**
> 第 7 课，**Part 3 · EDA 与数据预处理**。
> Lesson 7, **Part 3 · EDA & Preprocessing**.
>
> 模型只吃数字，文本怎么办？这一课讲**经典文本特征化**：清洗 → 词袋(BoW) → TF-IDF → n-gram。它是 NLP 的地基，也是很多实战任务（垃圾邮件、情感分析、工单分类）的第一版基线，至今在表格+文本混合任务里仍非常常用。
> Models only eat numbers — so how about text? This lesson covers **classic text featurization**: cleaning → bag-of-words (BoW) → TF-IDF → n-grams. It's the foundation of NLP and the first baseline for many real tasks (spam, sentiment, ticket routing), still widely used in tabular+text settings.
>
> 💼 **实战/面试视角**："文本怎么变特征 / TF-IDF 是什么 / 词袋的缺点" 是 NLP/数据岗常问。
> 💼 **Practical/interview angle:** "how to featurize text / what is TF-IDF / bag-of-words downsides" are common.

> 💡 **面试相关 / Interview-relevant**
> - "词袋(BoW) 怎么工作 / 它的缺点"（出镜率 ★★★★，丢失词序）
> - "TF-IDF 的含义 / 为什么比词频好"（★★★★★）
> - "n-gram 解决什么 / 维度代价"（★★★★）
> - "文本向量化为什么要放进 Pipeline（防泄漏）"（★★★★）

---

## 学习目标 / Learning Objectives

1. 做基础文本**清洗**（小写、去标点、规整空白）。
   Do basic text **cleaning** (lowercase, strip punctuation, normalize whitespace).
2. 理解**词袋 BoW** 及其"丢失词序"的缺点。
   Understand **bag-of-words** and its "loses word order" flaw.
3. 理解 **TF-IDF**：为什么稀有词更有区分力。
   Understand **TF-IDF**: why rare words are more discriminative.
4. 用 **n-gram** 找回部分词序（如 "click now"）。
   Use **n-grams** to recover some word order.
5. 用 **Pipeline** 把向量化+模型打包，自动防泄漏。
   Wrap vectorizer+model in a **Pipeline** to prevent leakage.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [文本清洗](#2)
3. [词袋 BoW ⭐](#3)
4. [TF-IDF ⭐](#4)
5. [区分性词 + n-gram ⭐](#5)
6. [Pipeline 防泄漏 + 实战 ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

把一句话变成数字，最朴素的想法：**统计每个词出现了几次**。整个语料的所有词构成"词表"，每条文本就是一个长度=词表大小的向量，第 j 位是"第 j 个词出现的次数"。这就是**词袋(bag-of-words)**——之所以叫"袋"，是因为它**只数词、不管顺序**（"狗咬人"和"人咬狗"的词袋一模一样，这正是它的核心缺点）。
The simplest way to turn a sentence into numbers: **count how often each word appears.** All words in the corpus form a "vocabulary"; each text becomes a vector of length = vocab size, where position j is "count of word j". This is **bag-of-words** — a "bag" because it **only counts words, ignoring order** ("dog bites man" and "man bites dog" have identical BoW vectors — its core flaw).

我们用一个 mini 垃圾短信数据集（spam vs ham）贯穿全课。
We use a mini spam-SMS dataset (spam vs ham) throughout.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

spam = [
    "WIN a free iPhone today click here now","URGENT you won a 1000 dollar prize call now",
    "Free entry weekly prize draw text WIN to 12345","Congratulations selected for free vacation",
    "Click link to claim your free reward now","Limited offer act now claim free gift card",
    "You have won cash prize call this number","Free ringtones text the number to win",
    "Claim your free holiday now urgent reply","Win big money click the link below now",
    "Exclusive offer free trial click now","Your account won a prize claim immediately",
]
ham = [
    "Hey are we still meeting for lunch tomorrow","Can you pick up some milk on your way home",
    "Happy birthday hope you have a great day","I will be late to the meeting sorry",
    "Did you watch the game last night","Lets grab coffee this weekend if free",
    "Mom called she wants you to call back","Running errands will see you at home tonight",
    "Thanks for the help yesterday really appreciate it","What time should I come over for dinner",
    "The report is due friday can we discuss","See you at the gym after work today",
]
df = pd.DataFrame({"text": spam+ham, "label": ["spam"]*len(spam)+["ham"]*len(ham)})
df = df.sample(frac=1, random_state=0).reset_index(drop=True)   # 打乱顺序 / shuffle
print(f"{len(df)} 条短信, {(df.label=='spam').sum()} spam / {(df.label=='ham').sum()} ham")
df.head(3)


<a id="2"></a>
## 2. 文本清洗 / Text Cleaning

向量化前先**规整文本**，减少无意义的差异：统一小写（让 "Free" 和 "free" 算同一个词）、去掉标点、规整空白。注意：清洗强度**取决于任务**——垃圾邮件里数字和 `$` 可能很重要，未必都删。
Before vectorizing, **normalize** the text to reduce meaningless variation: lowercase (so "Free" and "free" are the same word), strip punctuation, normalize whitespace. Note: cleaning aggressiveness **depends on the task** — for spam, digits and `$` may matter, so don't always strip them.


In [ ]:
import re

def clean(text):
    text = text.lower()                       # 统一小写: "Free"→"free"
    text = re.sub(r"[^a-z\s]", " ", text)     # 只保留字母和空白(演示用; 真实任务可能保留数字/$)
    return re.sub(r"\s+", " ", text).strip()  # 把多个空白压成一个, 去首尾空白

df["clean"] = df["text"].apply(clean)
for i in [0, 1]:
    print(f"原始 raw:   {df.text.iloc[i]}")
    print(f"清洗 clean: {df.clean.iloc[i]}\n")


<a id="3"></a>
## 3. 词袋 BoW ⭐ / Bag-of-Words

`CountVectorizer` 自动建词表并把每条文本变成词频向量。结果是一个**稀疏矩阵**（大部分元素是 0，因为一条短信只用到词表里很少的词），sklearn 只存非零元素以省内存。
`CountVectorizer` builds the vocabulary and turns each text into a word-count vector. The result is a **sparse matrix** (mostly zeros, since one message uses few of the vocabulary's words); sklearn stores only nonzeros to save memory.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X_count = cv.fit_transform(df["clean"])      # 返回稀疏矩阵 (n_docs × vocab_size)
vocab = cv.get_feature_names_out()           # 词表(按字母序排列)
print(f"词表大小 vocab size: {len(vocab)}")
print(f"特征矩阵 feature matrix: {X_count.shape} (稀疏, 只存非零 sparse)")
print(f"前 15 个词: {list(vocab[:15])}")

# 看第 0 条短信的词袋向量(只看非零部分) / one message's BoW vector (nonzeros)
row = X_count[0].toarray().ravel()           # toarray 把稀疏行转成普通数组
present = {vocab[j]: int(row[j]) for j in np.nonzero(row)[0]}   # 非零位置=出现的词
print(f"\n短信: '{df.clean.iloc[0]}'")
print(f"词袋(非零) BoW: {present}")


<a id="4"></a>
## 4. TF-IDF ⭐ / TF-IDF

词袋有个问题：常见词（the, you, to）出现次数多，但**几乎没有区分力**。**TF-IDF** 修正这一点：
BoW has a problem: common words (the, you, to) appear a lot but **carry little discriminative power**. **TF-IDF** corrects this:
- **TF（词频）**：词在该文档里出现得越多，越重要。
  **TF (term frequency):** the more a word appears in a document, the more important.
- **IDF（逆文档频率）**：词在**越少**的文档里出现，越有区分力（罕见词信息量大）。$\text{IDF} = \log\frac{\text{文档总数}}{\text{含该词的文档数}}$。
  **IDF (inverse document frequency):** the **fewer** documents a word appears in, the more discriminative (rare words are informative). $\text{IDF} = \log\frac{N}{\text{docs containing it}}$.
- **TF-IDF = TF × IDF**：既常见于本文、又罕见于全局的词得分最高。所以 "the" 被压低，"free"/"urgent" 被抬高。
  **TF-IDF = TF × IDF:** words frequent here but rare globally score highest. So "the" is down-weighted, "free"/"urgent" up-weighted.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["clean"])
# idf_ 存了每个词的 IDF 值; 越高=越罕见=越有区分力 / inspect IDF values
idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).sort_values(ascending=False)
print("IDF 最高的词(最罕见/最有区分力) highest IDF:")
print(idf.head(8).round(2).to_dict())
print("\nIDF 最低的词(最常见/最没区分力) lowest IDF:")
print(idf.tail(8).round(2).to_dict())
print("\n→ 'free'/'win'/'urgent' 等词 IDF 高被抬高; 'you'/'to' 等常见词 IDF 低被压低 — 符合直觉")


<a id="5"></a>
## 5. 区分性词 + n-gram ⭐ / Discriminative Words & n-grams

用 TF-IDF 的**类别均值差**可以找出"最像 spam / 最像 ham"的词。
The **class-mean difference** of TF-IDF reveals the most spam-like vs ham-like words.

**n-gram** 找回词序：词袋丢了顺序，但 "click now"、"free prize" 这种**短语**才是真正的垃圾信号。`ngram_range=(1,2)` 同时统计单词(unigram)和相邻词对(bigram)。代价是**特征数大增、更稀疏**——所以通常止于 bigram/trigram，并配 `min_df` 过滤罕见 n-gram。
**n-grams** recover order: BoW loses order, but phrases like "click now", "free prize" are the real spam signal. `ngram_range=(1,2)` counts both single words (unigrams) and adjacent pairs (bigrams). The cost is **far more, sparser features** — so usually stop at bigram/trigram and use `min_df` to filter rare ones.


In [ ]:
# 用 TF-IDF 类别均值差找最有区分力的词 / discriminative words via class-mean difference
spam_mask = (df.label == "spam").values
vocab = tfidf.get_feature_names_out()
spam_mean = np.asarray(X_tfidf[spam_mask].mean(axis=0)).ravel()    # spam 文档的平均 TF-IDF
ham_mean  = np.asarray(X_tfidf[~spam_mask].mean(axis=0)).ravel()
diff = pd.Series(spam_mean - ham_mean, index=vocab).sort_values()
print("最 'ham' 的词:", list(diff.head(5).index))
print("最 'spam' 的词:", list(diff.tail(5).index))

# unigram vs unigram+bigram / n-gram comparison
X1 = CountVectorizer(ngram_range=(1,1)).fit_transform(df.clean)
cv2 = CountVectorizer(ngram_range=(1,2)); X2 = cv2.fit_transform(df.clean)
print(f"\nunigram 特征数: {X1.shape[1]}")
print(f"unigram+bigram 特征数: {X2.shape[1]} (增多, 但抓住了'click now'等短语)")
print("部分 bigram:", [w for w in cv2.get_feature_names_out() if " " in w][:8])
print("⚠ n 越大特征越多越稀疏 → 通常止于 bigram/trigram, 配 min_df 过滤罕见 n-gram")


<a id="6"></a>
## 6. Pipeline 防泄漏 + 实战 ⭐ / Pipeline & In Practice

**关键纪律**：向量化器（CountVectorizer/TfidfVectorizer）的**词表和 IDF 必须只在训练集上学**。如果在全量数据上 fit，测试集的词汇统计就泄漏进了训练。**用 Pipeline 把向量化和模型打包**，交叉验证时它会自动在每折的训练部分 fit、在验证部分 transform——天然防泄漏。
**Key discipline:** the vectorizer's vocabulary and IDF must be **learned on the training set only**. Fitting on all data leaks the test set's vocabulary statistics into training. **Wrap the vectorizer and model in a Pipeline** — during cross-validation it auto-fits on each fold's train and transforms the validation part, preventing leakage by construction.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X, y = df["clean"], (df.label == "spam").astype(int)
# 三个端到端 pipeline: 向量化器 + 分类器打包 / vectorizer + classifier bundled
pipes = {
    "BoW + NaiveBayes":     Pipeline([("vec", CountVectorizer()), ("clf", MultinomialNB())]),
    "TF-IDF + NaiveBayes":  Pipeline([("vec", TfidfVectorizer()), ("clf", MultinomialNB())]),
    "TF-IDF(1,2) + LogReg": Pipeline([("vec", TfidfVectorizer(ngram_range=(1,2))),
                                      ("clf", LogisticRegression(max_iter=1000))]),
}
for name, pipe in pipes.items():
    print(f"{name:<26}: CV 准确率 = {cross_val_score(pipe, X, y, cv=4).mean():.1%}")
print("💡 Pipeline 保证 vectorizer 只在每折 train 上 fit → 自动防泄漏")

# 训练最终模型, 看逻辑回归学到的"垃圾词", 并预测新短信 / final model + predict
pipe = pipes["TF-IDF(1,2) + LogReg"].fit(X, y)
vec, clf = pipe.named_steps["vec"], pipe.named_steps["clf"]
coef = pd.Series(clf.coef_.ravel(), index=vec.get_feature_names_out())
print("\n逻辑回归学到的最强垃圾信号词(系数最大):", list(coef.sort_values(ascending=False).head(6).index))
for txt, p in zip(["free prize click now to win", "hey want to grab dinner tonight"],
                  pipe.predict([clean(t) for t in ["free prize click now to win", "hey want to grab dinner tonight"]])):
    print(f"  [{'spam' if p else 'ham'}] {txt}")


<a id="7"></a>
## 7. 小结 / Summary

```
文本→数字: 清洗(小写/去标点) → 词袋(数词频, 丢词序) → TF-IDF(TF×IDF, 压常见词抬稀有词) → n-gram(找回短语)
词袋缺点: 丢词序("狗咬人"="人咬狗"), 维度=词表大小, 高度稀疏
TF-IDF: 既常见于本文又罕见于全局的词得分最高; IDF=log(N/含该词文档数)
n-gram: (1,2)抓相邻词对; 特征暴增 → 止于 bigram/trigram + min_df 过滤
Pipeline: 向量化器只在每折 train 上 fit → 防泄漏(词表/IDF 不能见测试集)
这些是经典基线; 现代 NLP 用词向量/Transformer(后续部分)
```

### 💡 面试速查 / Interview cheat-sheet
1. **词袋只数词、丢词序**；维度=词表大小，高度稀疏。
   BoW counts words and loses order; dimension = vocab size, very sparse.
2. **TF-IDF = TF×IDF**：压低常见词、抬高稀有(有区分力)词。
   TF-IDF down-weights common words, up-weights rare discriminative ones.
3. **n-gram 找回短语**（click now），但特征暴增。
   n-grams recover phrases (click now) at a feature-count cost.
4. **向量化器只在训练集 fit**，用 Pipeline 自动保证。
   Fit the vectorizer on train only; Pipelines enforce it.
5. 这些是**经典基线**；现代 NLP 用词向量/Transformer。
   These are classic baselines; modern NLP uses embeddings/Transformers.

### 下一节 / Next
**3.8 图像特征基础**——图像怎么变成特征：像素展平、HOG 等传统特征，以及为什么深度学习用 CNN 自动学特征。
**3.8 Image Features** — turning images into features: pixel flattening, HOG, and why deep learning uses CNNs to learn features automatically.
